# PD03 · Integrar EVA con DIVIPOLA

[Índice y distribución de las cuatro horas](../ejercicios/README.md) · [Guía de pandas](../docs/manual-pandas.md)

**Tiempo de trabajo: 70 minutos.** El objetivo es comprobar una unión territorial y calcular un rendimiento con el denominador adecuado.

| Símbolo | Uso |
|:---:|---|
| 📘 | Concepto o función de apoyo. |
| ✅ | Evidencia que debe quedar visible. |
| ⚠️ | Límite de interpretación. |

Esta plantilla se encuentra en `ejercicios/PD03_integracion_territorial.ipynb`. Antes de comenzar, se crea una copia en `soluciones/PD03_integracion_territorial.ipynb` dentro del repositorio de cada estudiante. El desarrollo y la entrega se realizan en esa copia; la plantilla de `ejercicios/` se conserva como enunciado. En JupyterLab se puede duplicar el archivo y mover la copia a `soluciones/`, manteniendo el nombre indicado; si ya existe una solución, se continúa sobre ella. Las instrucciones se presentan en celdas Markdown; las celdas de código marcadas como **Desarrollo** contienen el espacio de trabajo. Cada actividad incluye una celda para explicar el resultado.

Se utiliza el kernel del entorno `.venv` del curso. Los CSV ya deben estar descargados mediante `kit/00_datos.py`. La celda inicial prepara las rutas y carga únicamente las entradas de este ejercicio; el desarrollo del análisis corresponde al estudiante. Este notebook se ejecuta de forma independiente.

**Apoyo en el manual:** secciones 4, 7, 9, 10 y la receta de razón agregada de la sección 11.

## Ubicación de trabajo: la raíz del repositorio

📘 **La raíz del repositorio** es la carpeta principal de la copia del proyecto. En la instalación del curso se llama `big-data-postgraduate` y contiene `README.md`, `kit/`, `ejercicios/` y `soluciones/`:

```text
bigdata/
└── big-data-postgraduate/       ← raíz del repositorio
    ├── README.md
    ├── kit/
    ├── ejercicios/              ← enunciados y plantillas
    └── soluciones/              ← notebooks resueltos
```

**Desde la terminal de Ubuntu en WSL.** `pwd` muestra la carpeta actual, `cd` cambia de carpeta y `ls` muestra su contenido. Los siguientes comandos se ejecutan en la terminal, no en una celda Python:

```bash
pwd
cd /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate
pwd
ls
```

✅ Después de `cd`, el segundo `pwd` debe mostrar `/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate`. En la salida de `ls` deben aparecer `README.md`, `kit`, `ejercicios` y `soluciones`. La carpeta `bigdata` es la carpeta contenedora; la raíz del proyecto es su subcarpeta `big-data-postgraduate`.

La ruta anterior corresponde a la configuración de la guía WSL. Si el repositorio se clonó en otra ubicación, se sustituye por la ruta real de esa copia. Si `cd` informa que la carpeta no existe, se revisa esa ubicación antes de continuar.

**Desde una subcarpeta conocida.** `..` significa «carpeta superior». Se comprueba primero la ubicación con `pwd` y se utiliza la alternativa correspondiente:

| Ubicación actual, relativa al repositorio | Comando para volver a la raíz |
|---|---|
| `ejercicios/`, `soluciones/` o `kit/` | `cd ..` |
| `kit/Notebooks/` | `cd ../..` |

Al llegar a la raíz, se puede activar el entorno y abrir JupyterLab desde la terminal:

```bash
source .venv/bin/activate
jupyter lab
```

**Dentro de un notebook ya abierto.** La terminal y el kernel de Python mantienen sus propios directorios de trabajo. La celda de carga siguiente localiza la raíz y la guarda en `REPO`, aunque el notebook esté abierto desde `soluciones/` o `ejercicios/`. Después de ejecutar esa celda, el siguiente bloque puede ejecutarse en una celda Python para comprobar la ubicación y cambiar el directorio del kernel a la raíz:

```python
import os
from pathlib import Path

print("Directorio antes del cambio:", Path.cwd())
print("Raíz detectada:", REPO)
os.chdir(REPO)
print("Directorio después del cambio:", Path.cwd())
```

✅ `Path.cwd()` debe coincidir con `REPO` después del cambio. Esta operación cambia la base para interpretar rutas relativas; el archivo del notebook permanece en `soluciones/`. La carga de datos de estos ejercicios también funciona sin ese cambio porque construye las rutas a partir de `REPO`.

## Preparación

Se ejecuta la siguiente celda y se comprueba que el intérprete pertenece al entorno del curso y que pandas corresponde a la versión 2.2.3. Las rutas se calculan desde el repositorio, sin introducir rutas personales de Windows, WSL o macOS.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

actual = Path.cwd().resolve()
REPO = next(
    (ruta for ruta in [actual, *actual.parents]
     if (ruta / "kit/fuentes.json").is_file()
     and (ruta / "kit/00_datos.py").is_file()),
    None,
)
if REPO is None:
    raise FileNotFoundError("El notebook debe abrirse dentro del repositorio del curso")

RAW = REPO / "kit/data/raw"
OUT = REPO / "kit/salidas/soluciones/PD03"
archivos = ['eva.csv', 'divipola.csv']
faltantes = [nombre for nombre in archivos if not (RAW / nombre).is_file()]
if faltantes:
    raise FileNotFoundError(f"Faltan CSV en {RAW}: {faltantes}")
OUT.mkdir(parents=True, exist_ok=True)

opciones = dict(sep=",", encoding="utf-8-sig", dtype=str, keep_default_na=False)
eva = pd.read_csv(RAW / "eva.csv", **opciones)
divipola = pd.read_csv(RAW / "divipola.csv", **opciones)


print("Python:", sys.version.split()[0])
print("Intérprete:", sys.executable)
print("pandas:", pd.__version__)
print("Repositorio:", REPO)
print("Salidas:", OUT)

Python: 3.12.15
Intérprete: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/.venv/bin/python
pandas: 2.2.3
Repositorio: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate
Salidas: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD03


## 1. Dimensión territorial · 10 minutos

Crear `codigo_municipio` desde `Código Municipio` de DIVIPOLA. Comprobar cinco cifras y unicidad. Contar los valores de `Tipo: Municipio / Isla / Área no municipalizada`. Construir `dimension` con código, nombre departamental, nombre municipal y clasificación territorial.

📘 **Apoyo:** `.str.strip`, `.str.fullmatch(r"[0-9]{5}")`, `is_unique` y `value_counts`.

✅ **Comprobación:** La dimensión conserva las 1.122 entidades y sus tres tipos. La columna de código es única y no contiene vacíos.

In [2]:
from IPython.display import Markdown
import numpy as np
fmt = lambda n: f"{n:,}".replace(",", ".")

# Columnas de DIVIPOLA (se detectan por nombre; se imprimen para verificar)
col_tipo = "Tipo: Municipio / Isla / Área no municipalizada"
col_depto = next(c for c in divipola.columns if "departamento" in c.lower() and "nombre" in c.lower())
col_mun = next(c for c in divipola.columns if "municipio" in c.lower() and "nombre" in c.lower())
print("Departamento:", col_depto, "| Municipio:", col_mun)

divipola["codigo_municipio"] = divipola["Código Municipio"].str.strip()

formato_ok = divipola["codigo_municipio"].str.fullmatch(r"[0-9]{5}")
n_mal = int((~formato_ok).sum())
n_vacios = int(divipola["codigo_municipio"].eq("").sum())
es_unico = divipola["codigo_municipio"].is_unique
print("Códigos de cinco cifras:", int(formato_ok.sum()), "de", len(divipola))
print("Códigos mal formados:", n_mal, "| vacíos:", n_vacios, "| únicos:", es_unico)
assert n_mal == 0 and n_vacios == 0 and es_unico

tipos = divipola[col_tipo].value_counts()
display(tipos.to_frame("entidades"))

dimension = divipola[["codigo_municipio", col_depto, col_mun, col_tipo]].copy()
dimension.columns = ["codigo_municipio", "departamento", "municipio", "tipo_territorial"]
assert len(dimension) == len(divipola)
display(dimension.head())



Departamento: Nombre Departamento | Municipio: Nombre Municipio
Códigos de cinco cifras: 1122 de 1122
Códigos mal formados: 0 | vacíos: 0 | únicos: True


,entidades
Tipo: Municipio / Isla / Área no municipalizada,
Municipio,1103
Área no municipalizada,18
Isla,1


,codigo_municipio,departamento,municipio,tipo_territorial
0,05001,ANTIOQUIA,MEDELLÍN,Municipio
1,05002,ANTIOQUIA,ABEJORRAL,Municipio
2,05004,ANTIOQUIA,ABRIAQUÍ,Municipio
3,05021,ANTIOQUIA,ALEJANDRÍA,Municipio
4,05030,ANTIOQUIA,AMAGÁ,Municipio


**Interpretación del resultado:**

La dimensión territorial tiene 1.122 entidades (el corte esperado es 1.122) y conserva sus 3 tipos: Municipio (1.103); Área no municipalizada (18); Isla (1). Todos los códigos tienen exactamente cinco cifras, ninguno está vacío y ninguno se repite, así que codigo_municipio sirve como llave única para unir. Esto último es clave porque, si la dimensión tuviera códigos repetidos, al unirla con EVA se duplicarían filas y se inflarían las sumas. Se dejó el código como texto para no perder ceros a la izquierda (por ejemplo 05001 de Medellín).

## 2. Códigos EVA sin correspondencia · 15 minutos

Preparar `codigo_municipio` desde `Código Dane municipio` y `anio` desde `Año` de EVA. Conservar originales; los códigos vacíos o mal formados deben quedar identificados para revisión. Completar ceros solo en textos numéricos válidos de menor longitud. Crear combinaciones distintas de código y año y realizar un anti-join frente a dimension. Guardar el resultado en `sin_correspondencia`.

📘 **Apoyo:** `.str.fullmatch`, `.where`, `.str.zfill`, `drop_duplicates` y `merge(how="left", indicator=True, validate="many_to_one")`.

✅ **Comprobación:** El código municipal es la clave de unión. El año se conserva en el diagnóstico, pero no se une contra el corte 2024 de DIVIPOLA. El listado se muestra aunque quede vacío.

In [3]:
cod = eva["Código Dane municipio"].str.strip()
valido = cod.str.fullmatch(r"[0-9]{5}")
corto = cod.str.fullmatch(r"[0-9]{1,4}")

# Solo se completan con ceros los textos numéricos válidos de menos de cinco cifras
eva["codigo_municipio"] = cod.where(valido, cod.where(corto).str.zfill(5))
eva["codigo_para_revision"] = eva["codigo_municipio"].isna()   # vacíos o mal formados
eva["anio"] = pd.to_numeric(eva["Año"].str.strip(), errors="coerce").astype("Int64")

n_completados = int((eva["codigo_municipio"].notna() & ~valido).sum())
n_revision = int(eva["codigo_para_revision"].sum())
print("Códigos completados con ceros:", n_completados)
print("Códigos vacíos o mal formados (para revisión):", n_revision)
display(eva.loc[eva["codigo_para_revision"], ["Código Dane municipio", "codigo_municipio"]].drop_duplicates().head(5))

combinaciones = eva.loc[eva["codigo_municipio"].notna(), ["codigo_municipio", "anio"]].drop_duplicates()
cruce = combinaciones.merge(
    dimension[["codigo_municipio"]], on="codigo_municipio",
    how="left", indicator=True, validate="many_to_one",
)
sin_correspondencia = (
    cruce.loc[cruce["_merge"].eq("left_only"), ["codigo_municipio", "anio"]]
    .sort_values(["codigo_municipio", "anio"]).reset_index(drop=True)
)
print("Combinaciones código-año:", len(combinaciones))
print("Combinaciones sin correspondencia:", len(sin_correspondencia))
display(sin_correspondencia)   # se muestra aunque esté vacío

filas_afectadas = int(eva["codigo_municipio"].isin(sin_correspondencia["codigo_municipio"]).sum())
codigos_afectados = sin_correspondencia["codigo_municipio"].nunique()
if len(sin_correspondencia) == 0:
    texto_sc = "Todas las combinaciones de código y año válidas encontraron su municipio en la dimensión, así que el anti-join salió vacío."
else:
    texto_sc = f"El anti-join arrojó {fmt(len(sin_correspondencia))} combinaciones sin correspondencia, que corresponden a {fmt(codigos_afectados)} códigos distintos y afectan {fmt(filas_afectadas)} filas de EVA."


Códigos completados con ceros: 0
Códigos vacíos o mal formados (para revisión): 0


,Código Dane municipio,codigo_municipio


Combinaciones código-año: 7691
Combinaciones sin correspondencia: 0


,codigo_municipio,anio


**Interpretación del resultado:**

En EVA se completaron con ceros 0 códigos numéricos de menos de cinco cifras y quedaron 0 códigos vacíos o mal formados marcados en codigo_para_revision (no se inventó ningún código). Se obtuvieron 7.691 combinaciones distintas de código y año. Todas las combinaciones de código y año válidas encontraron su municipio en la dimensión, así que el anti-join salió vacío. La llave de la unión es solo el código municipal; el año se conserva en el diagnóstico para saber en qué años aparece cada código, pero no se une contra DIVIPOLA porque ese catálogo es un corte de 2024 y no tiene una versión por año. Un código sin correspondencia no es necesariamente un error: puede ser un municipio nuevo, uno que cambió de código, o un código mal digitado.

## 3. Selección para el indicador · 10 minutos

Convertir `Área cosechada` y `Producción` a `area_ha` y `produccion_t`. Seleccionar primero `Cultivo == "Papa"` y año 2024 en `papa_2024`. De ese conjunto, conservar en `apta` las filas con área positiva y producción no negativa. Contar las filas excluidas del indicador dentro de papa_2024.

📘 **Apoyo:** `pd.to_numeric`, `.eq`, `.gt`, `.ge`, `.loc` y `.copy()`.

✅ **Comprobación:** El total de papa_2024 es igual a las filas aptas más las excluidas del indicador. La ausencia de una medida no se reemplaza por cero.

In [4]:
eva["area_ha"] = pd.to_numeric(eva["Área cosechada"].str.strip(), errors="coerce")
eva["produccion_t"] = pd.to_numeric(eva["Producción"].str.strip(), errors="coerce")

papa_2024 = eva.loc[eva["Cultivo"].str.strip().eq("Papa") & eva["anio"].isin([2024])].copy()
apta = papa_2024.loc[papa_2024["area_ha"].gt(0) & papa_2024["produccion_t"].ge(0)].copy()
excluidas = len(papa_2024) - len(apta)

motivos = pd.Series({
    "área ausente": int(papa_2024["area_ha"].isna().sum()),
    "área menor o igual a cero": int((papa_2024["area_ha"].notna() & papa_2024["area_ha"].le(0)).sum()),
    "producción ausente": int(papa_2024["produccion_t"].isna().sum()),
    "producción negativa": int((papa_2024["produccion_t"].notna() & papa_2024["produccion_t"].lt(0)).sum()),
}, name="filas")
display(motivos.to_frame())

conciliacion = pd.DataFrame({
    "concepto": ["papa_2024", "apta", "excluidas del indicador"],
    "filas": [len(papa_2024), len(apta), excluidas],
})
display(conciliacion)
assert len(apta) + excluidas == len(papa_2024)




,filas
área ausente,0
área menor o igual a cero,52
producción ausente,0
producción negativa,0


,concepto,filas
0,papa_2024,831
1,apta,779
2,excluidas del indicador,52


**Interpretación del resultado:**
Para papa en 2024 hay 831 filas. De ellas, 779 son aptas para calcular el indicador (área positiva y producción no negativa) y 52 quedan excluidas; ambas cifras suman el total de papa_2024. Por motivos: 0 con área ausente, 52 con área cero o negativa, 0 con producción ausente y 0 con producción negativa; una fila puede tener más de un motivo, por eso esa suma puede superar las excluidas. Seleccioné primero cultivo y año y después apliqué la validez de las medidas para poder contar con claridad qué se descarta. No reemplacé ninguna medida ausente por cero: una producción de cero sería un dato inventado y sin área positiva no se puede dividir.


## 4. Rendimiento agregado · 15 minutos

Agrupar apta por código municipal, cultivo, estado físico del cultivo y año. Crear `rendimiento` con cantidad de registros, suma de producción y suma de área cosechada. Calcular `rendimiento_t_ha` dividiendo ambas sumas.

📘 **Apoyo:** `groupby(..., as_index=False, dropna=False)`, `agg`, `size` y `sum`.

✅ **Comprobación:** Cada grupo tiene área total positiva. La suma de los conteos por grupo coincide con len(apta). El indicador se calcula como suma de producción / suma de área, conservando cultivo y estado físico en la agrupación.

In [5]:
col_estado = next(c for c in eva.columns if "estado" in c.lower())
print("Columna de estado físico:", col_estado)

claves = ["codigo_municipio", "Cultivo", col_estado, "anio"]
rendimiento = apta.groupby(claves, as_index=False, dropna=False).agg(
    registros=("area_ha", "size"),
    produccion_total_t=("produccion_t", "sum"),
    area_total_ha=("area_ha", "sum"),
)
rendimiento["rendimiento_t_ha"] = rendimiento["produccion_total_t"] / rendimiento["area_total_ha"]

assert rendimiento["area_total_ha"].gt(0).all()
assert rendimiento["registros"].sum() == len(apta)
print("Grupos:", len(rendimiento), "| registros agrupados:", int(rendimiento["registros"].sum()), "| len(apta):", len(apta))
display(rendimiento["rendimiento_t_ha"].describe().to_frame("rendimiento_t_ha").round(2))
display(rendimiento.sort_values("rendimiento_t_ha", ascending=False).head(5).round(2))

global_pond = rendimiento["produccion_total_t"].sum() / rendimiento["area_total_ha"].sum()
promedio_simple = rendimiento["rendimiento_t_ha"].mean()


Columna de estado físico: Estado físico del cultivo
Grupos: 292 | registros agrupados: 779 | len(apta): 779


,rendimiento_t_ha
count,292.00
mean,17.98
std,6.59
min,3.00
25%,14.36
50%,18.00
75%,21.72
max,48.76


,codigo_municipio,Cultivo,Estado físico del cultivo,anio,registros,produccion_total_t,area_total_ha,rendimiento_t_ha
25,05686,Papa,En fresco,2024,4,56956.0,1168.0,48.76
29,05887,Papa,En fresco,2024,2,6435.0,143.0,45.00
184,25736,Papa,En fresco,2024,2,49122.0,1286.0,38.20
23,05664,Papa,En fresco,2024,4,36915.0,991.5,37.23
275,68861,Papa,En fresco,2024,1,2940.0,84.0,35.00


**Interpretación del resultado:**

Las filas aptas quedaron agrupadas en 292 grupos de municipio, cultivo, estado físico y año, y la suma de registros por grupo (779) coincide con len(apta). Todos los grupos tienen área total positiva, así que ninguna división es por cero. El rendimiento de cada grupo es producción total dividida entre área total, y va de 3.00 a 48.76 t/ha, con mediana de 18.00. El rendimiento global ponderado es 22.19 t/ha, mientras que el promedio simple de los rendimientos de los grupos da 17.98; la diferencia muestra que cada forma de calcular da un resultado distinto. Se dejó dropna=False para no perder grupos con alguna clave ausente, y se mantuvo el estado físico dentro de la agrupación porque mezclar estados distintos haría que el indicador no fuera comparable.

## 5. Unión con nombres y clasificación · 10 minutos

Relacionar rendimiento con dimension mediante un left merge por codigo_municipio. Declarar `validate="many_to_one"` e incorporar un indicador de correspondencia. Guardar el resultado en `rendimiento_municipal` y mostrar los códigos sin nombre territorial, si existen.

📘 **Apoyo:** `merge`, `validate`, `indicator` y `len`.

✅ **Comprobación:** La cantidad de filas antes y después de la unión debe ser idéntica. Un fallo de cardinalidad debe investigarse; no se resuelve eliminando arbitrariamente claves de la dimensión.

In [6]:
filas_antes = len(rendimiento)
rendimiento_municipal = rendimiento.merge(
    dimension, on="codigo_municipio", how="left",
    validate="many_to_one", indicator="correspondencia",
)
rendimiento_municipal["correspondencia"] = rendimiento_municipal["correspondencia"].astype(str)

assert len(rendimiento_municipal) == filas_antes
print("Filas antes:", filas_antes, "| después:", len(rendimiento_municipal))

cobertura = rendimiento_municipal["correspondencia"].value_counts()
display(cobertura.to_frame("filas"))

sin_nombre = (rendimiento_municipal.loc[rendimiento_municipal["correspondencia"].eq("left_only"), ["codigo_municipio"]]
              .drop_duplicates())
print("Códigos sin nombre territorial:", len(sin_nombre))
display(sin_nombre)   # se muestra aunque esté vacío
display(rendimiento_municipal.head())

con_nombre = int(cobertura.get("both", 0))


Filas antes: 292 | después: 292


,filas
correspondencia,
both,292


Códigos sin nombre territorial: 0


,codigo_municipio


,codigo_municipio,Cultivo,Estado físico del cultivo,anio,registros,produccion_total_t,area_total_ha,rendimiento_t_ha,departamento,municipio,tipo_territorial,correspondencia
0,05001,Papa,En fresco,2024,2,1280.0,80.0,16.000000,ANTIOQUIA,MEDELLÍN,Municipio,both
1,05002,Papa,En fresco,2024,2,1650.0,165.0,10.000000,ANTIOQUIA,ABEJORRAL,Municipio,both
2,05004,Papa,En fresco,2024,4,15.0,5.0,3.000000,ANTIOQUIA,ABRIAQUÍ,Municipio,both
3,05079,Papa,En fresco,2024,2,268.0,34.0,7.882353,ANTIOQUIA,BARBOSA,Municipio,both
4,05086,Papa,En fresco,2024,3,10550.0,905.0,11.657459,ANTIOQUIA,BELMIRA,Municipio,both


**Interpretación del resultado:**

El left merge conservó las 292 filas de rendimiento (después de la unión hay 292), y con validate="many_to_one" pandas habría dado error si un código se repitiera en la dimensión. 292 filas (100.0 %) encontraron nombre y clasificación territorial y 0 códigos quedaron sin nombre. Usé left merge para no perder rendimientos de municipios que no aparecen en el catálogo: si hubiera usado un inner join, esas filas desaparecerían sin avisar. Si la cardinalidad hubiera fallado, habría que investigar el origen del duplicado y no borrar claves de la dimensión al azar.

## 6. Exportación e interpretación · 10 minutos

Exportar rendimiento_municipal a `OUT / "rendimiento_municipal.csv"` y sin_correspondencia a `OUT / "codigos_sin_correspondencia.csv"`, ambos sin índice. Explicar por qué se dividen sumas en lugar de promediar rendimientos individuales y qué limitación introduce comparar EVA con un catálogo territorial de corte 2024.

📘 **Apoyo:** `to_csv` y los resultados de las actividades anteriores.

✅ **Comprobación:** El archivo de códigos conserva sus cabeceras aunque no tenga filas. La explicación distingue una coincidencia de códigos de una equivalencia histórica de límites.

In [7]:
ruta_rend = OUT / "rendimiento_municipal.csv"
ruta_sin = OUT / "codigos_sin_correspondencia.csv"
rendimiento_municipal.to_csv(ruta_rend, index=False)
sin_correspondencia.to_csv(ruta_sin, index=False)

print("rendimiento_municipal.csv:", ruta_rend.is_file(), "| filas:", len(rendimiento_municipal))
print("codigos_sin_correspondencia.csv:", ruta_sin.is_file(), "| cabeceras:", list(pd.read_csv(ruta_sin, nrows=0).columns))

display(Markdown(f"""**Interpretación del resultado:** Se exportaron `rendimiento_municipal.csv` ({fmt(len(rendimiento_municipal))} filas) y `codigos_sin_correspondencia.csv` ({fmt(len(sin_correspondencia))} filas, con sus cabeceras aunque estuviera vacío).

**Por qué se dividen sumas.** El rendimiento de un municipio se calcula como suma de producción entre suma de área, y no como el promedio de los rendimientos de cada registro. Un promedio de razones le da el mismo peso a un lote de media hectárea que a uno de cien, y distorsiona el resultado; la razón de sumas pondera cada registro por su área, que es el denominador real. Aquí se nota: el rendimiento global ponderado fue {global_pond:.2f} t/ha y el promedio simple de los grupos {promedio_simple:.2f} t/ha.

**Limitación del corte 2024.** Que un código de EVA aparezca en DIVIPOLA solo prueba que los códigos coinciden, no que el municipio tenga los mismos límites que en el año del registro. Los municipios pueden crearse, fusionarse o cambiar de código con el tiempo, y como la dimensión es un corte de 2024 no hay forma de comprobar esa estabilidad para otros años. Además, el rendimiento es válido solo para el cultivo y el estado físico seleccionados y no debe sumarse con el de otros cultivos."""))

rendimiento_municipal.csv: True | filas: 292
codigos_sin_correspondencia.csv: True | cabeceras: ['codigo_municipio', 'anio']


**Interpretación del resultado:** Se exportaron `rendimiento_municipal.csv` (292 filas) y `codigos_sin_correspondencia.csv` (0 filas, con sus cabeceras aunque estuviera vacío).

**Por qué se dividen sumas.** El rendimiento de un municipio se calcula como suma de producción entre suma de área, y no como el promedio de los rendimientos de cada registro. Un promedio de razones le da el mismo peso a un lote de media hectárea que a uno de cien, y distorsiona el resultado; la razón de sumas pondera cada registro por su área, que es el denominador real. Aquí se nota: el rendimiento global ponderado fue 22.19 t/ha y el promedio simple de los grupos 17.98 t/ha.

**Limitación del corte 2024.** Que un código de EVA aparezca en DIVIPOLA solo prueba que los códigos coinciden, no que el municipio tenga los mismos límites que en el año del registro. Los municipios pueden crearse, fusionarse o cambiar de código con el tiempo, y como la dimensión es un corte de 2024 no hay forma de comprobar esa estabilidad para otros años. Además, el rendimiento es válido solo para el cultivo y el estado físico seleccionados y no debe sumarse con el de otros cultivos.

**Interpretación del resultado:**

_Pendiente de completar con evidencia de las tablas o controles anteriores._

## Entrega

El notebook completado se conserva como `soluciones/PD03_integracion_territorial.ipynb`. Las salidas regenerables se guardan en `kit/salidas/soluciones/PD03/`.

✅ Deben quedar visibles el control de la dimensión, el anti-join, el conteo de exclusiones, la conciliación de grupos y la conservación de filas en el merge.

⚠️ Una coincidencia de códigos no prueba estabilidad de límites municipales a través del tiempo. El rendimiento agregado se interpreta para el cultivo y estado físico seleccionados; no debe sumarse con rendimientos de otros cultivos.